# Espectrogramas reales y Griffin-Lim

Notebook opcional del **módulo 15** del curso (en la app: `uv run dev` y la ruta `#/modulo/audio-senales`). Usa modelos reales; el módulo interactivo explica la teoría con NumPy.

> **Requisitos:** desde la raíz del repositorio, `uv sync --group notebooks` y abre este notebook con `uv run --group notebooks jupyter lab`. La primera ejecución descarga los modelos de Hugging Face.

In [ ]:
import librosa
import librosa.display
import matplotlib.pyplot as plt
import numpy as np

y, sr = librosa.load(librosa.ex("trumpet"), sr=16000)
print(len(y), "muestras a", sr, "Hz")

In [ ]:
S = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=400, hop_length=160, n_mels=80)
librosa.display.specshow(librosa.power_to_db(S, ref=np.max), sr=sr, hop_length=160, x_axis="time", y_axis="mel")
plt.colorbar(format="%+2.0f dB")
plt.show()

In [ ]:
from transformers import WhisperFeatureExtractor

fe = WhisperFeatureExtractor.from_pretrained("openai/whisper-small")
x = fe(y, sampling_rate=16000, return_tensors="np")["input_features"]
print("entrada de Whisper:", x.shape)   # (1, 80, 3000): siempre 30 s

In [ ]:
magnitud = np.abs(librosa.stft(y, n_fft=512, hop_length=128))
reconstruida = librosa.griffinlim(magnitud, n_iter=32, hop_length=128)
print("correlación con la original:", np.corrcoef(y[: len(reconstruida)], reconstruida)[0, 1].round(3))